<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

%pip -q install duckdb huggingface_hub

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


## 1. Unit of analysis + time window


One row = one content item (content_hash_id) within one client (client_hash_id), summarized over a defined feature window, for Lane 2 (Refresh/Content Opportunity Scoring).

Anchor month: mid-panel month 2026-03 — treated as "today" for this contract. I use a mid-panel month, not the final month, because fact_daily_sample covers only June 2026 (the sealed outcome window for any past→future label), so developing against it would mean testing on the future I'm trying to predict.
Feature window: the 90 days ending 2026-03-31 (2025-12-31 to 2026-03-31).
Label window: the following 30 days, 2026-04-01 to 2026-04-30 — kept strictly after the feature window so no row's features are computed from dates inside its own label window.
Label/proxy: is_declining_future — impressions in the label window drop more than 20% versus the feature window's average 30-day pace. This is a genuine future-outcome label (unlike the starter dataset's trend_direction, which is a current-state bucket), matching the stronger version flagged as a goal in w01/w02.
One thing deliberately excluded: any FlyRank product-computed field (health_score, priority_score, action_type, refresh flags). These aren't shipped in the release anyway, but I state the exclusion explicitly — if I ever rebuild one, it goes in as a comparison baseline, never as a model feature, per Section 4/13 of the lane guide (avoiding a circular result).

In [2]:
from datetime import date

anchor_date = date(2026, 3, 31)
feature_window_start = date(2025, 12, 31)
feature_window_end = anchor_date
label_window_start = date(2026, 4, 1)
label_window_end = date(2026, 4, 30)

print(f"Feature window: {feature_window_start} to {feature_window_end}")
print(f"Label window:   {label_window_start} to {label_window_end}")
assert label_window_start > feature_window_end, "Label window overlaps feature window!"
print(f"\nNo overlap confirmed: label window starts {(label_window_start - feature_window_end).days} day(s) after feature window ends")

Feature window: 2025-12-31 to 2026-03-31
Label window:   2026-04-01 to 2026-04-30

No overlap confirmed: label window starts 1 day(s) after feature window ends


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `imp_prev90`, `clk_prev90`, `pos_avg_prev90`, `ctr_prev90`, `days_active_prev90` | All computed only from `fact_content_daily_performance` for dates ≤ 2026-03-31 — knowable before the label window begins |
| **Label** | `is_declining_future` (impressions in 2026-04-01→2026-04-30 vs. feature-window pace) | The real future outcome I'm trying to rank pages by risk of |
| **Context** | `client_hash_id`, `content_hash_id`, `dim_clients.gsc_data_start`, `dim_clients.ga4_data_start` | Used for grouping and to check history availability before trusting a row, not as model inputs |
| **Excluded** | `health_score`, `priority_score`, `action_type`, refresh flags (not in release); raw query/URL/title fields (not in release); `visible_queries`, `rare_share`, `anon_share`, `top_query_share` from `fact_content_query_90d` — Query D confirmed this table is a single fixed window (2026-04-02 to 2026-06-30) that overlaps and postdates my March-anchored label window, making it a hidden leak risk; GA4-derived fields where `ga4_data_available IS NOT TRUE` (only 4.2% of March rows have GA4) | Product-decision fields risk circularity; raw fields are excluded by design; the query-mix table's fixed window leaks future information; GA4 is too sparse in this slice to trust |

In [3]:
print("fact_content_daily_performance columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()[['column_name', 'column_type']])

print("\nfact_content_query_90d columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']}").df()[['column_name', 'column_type']])

print("\ndim_clients columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']}").df()[['column_name', 'column_type']])

fact_content_daily_performance columns:
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22               

## 3. Verify it with queries (grain, counts, missing values, windows)

**Query A** confirms the grain: for one sample content_hash_id + client_hash_id pair, multiple daily rows collapse into exactly one row after aggregation.

In [4]:
# Query A — grain check
sample_pair = con.sql(f"""
    SELECT client_hash_id, content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    LIMIT 1
""").df().iloc[0]

daily_rows = con.sql(f"""
    SELECT report_date, gsc_impressions, gsc_clicks
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    ORDER BY report_date
""").df()
print(f"Daily rows for this content x client pair in March: {len(daily_rows)}")

aggregated = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n_days, SUM(gsc_impressions) AS imp_total
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2
""").df()
print(f"After aggregation: {len(aggregated)} row(s) — confirms one row per content x client")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Daily rows for this content x client pair in March: 31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

After aggregation: 1 row(s) — confirms one row per content x client


**Query B** shows the slice's row count and date span for the mid-panel month.

In [5]:
# Query B — slice row count and date span
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print(slice_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows   min_date   max_date  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


**Query C** filters with IS TRUE on GA4 availability, per the assignment's requirement, and shows how many rows survive.

In [6]:
# Query C — availability filter with IS TRUE
avail = con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
avail['pct_ga4_available'] = (avail['ga4_available_rows'] / avail['total_rows'] * 100).round(1)
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  pct_ga4_available
0     9841378            413966.0                4.2


**Query D** for the Window check

In [7]:
window_check = con.sql(f"""
    SELECT MIN(window_start) AS min_start, MAX(window_start) AS max_start,
           MIN(window_end) AS min_end, MAX(window_end) AS max_end,
           COUNT(DISTINCT window_start) AS n_distinct_starts,
           COUNT(DISTINCT window_end) AS n_distinct_ends
    FROM {TABLES['fact_query_90d']}
""").df()
print(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   min_start  max_start    min_end    max_end  n_distinct_starts  \
0 2026-04-02 2026-04-02 2026-06-30 2026-06-30                  1   

   n_distinct_ends  
0                1  


**Finding:** fact_content_query_90d is a single fixed window (2026-04-02 to 2026-06-30), not recomputed per anchor month. That window starts one day after my feature window ends and extends two months past my label window — so visible_queries and rare_share were never knowable on 2026-03-31. This is a second, less obvious leak than the deliberate imp_next30 one below: it looks like an ordinary feature until the window bounds are actually checked. Correction: I drop both fields and replace them with two additional safe features from fact_daily, the table Queries A–C already verified as month-aligned.

**Five features**, each knowable strictly before the label window (2026-04-01) begins:

- imp_prev90 — knowable at the decision moment because it only sums impressions through 2026-03-31.
- clk_prev90 — same window, knowable for the same reason.
- pos_avg_prev90 — average search position through 2026-03-31, no future dates touched.
- visible_queries — count of distinct queries a page ranks for, from the 90-day query-mix table ending in the feature window.
- rare_share — share of impressions from rare/anonymized queries, same 90-day query table, no overlap with the label window.


In [8]:
# Build the five-feature frame for the mid-panel month
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev90,
           SUM(gsc_clicks)      AS clk_prev90,
           AVG(gsc_avg_position) AS pos_avg_prev90
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev90 >= 100
""").df()

q = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share)       AS rare_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

feature_frame = feat.merge(q, on='content_hash_id', how='left')
print(f"{len(feature_frame):,} rows, 5 features")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,372 rows, 5 features


,client_hash_id,content_hash_id,imp_prev90,clk_prev90,pos_avg_prev90,visible_queries,rare_share
0,client_3ffa76342f366962,content_b89167cd03d6ffc1,255.0,11.0,2.955310,NaN,NaN
1,client_e547b89c05043229,content_2e296120acb03e93,4212.0,0.0,50.856976,43.0,0.042632
2,client_e547b89c05043229,content_516b7c0e8eec0cef,4966.0,0.0,49.090790,13.0,0.112391
3,client_e547b89c05043229,content_38b6c1a9aa29f801,4654.0,0.0,47.352218,129.0,0.069119
4,client_e547b89c05043229,content_2ffd36f2a70be7e3,2211.0,0.0,36.764236,30.0,0.061182


Checking whether these features are actually safe: visible_queries and rare_share come from fact_content_query_90d. Before trusting them, I check whether that table is month-aligned like fact_daily or a fixed snapshot.

In [9]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev90,
           SUM(gsc_clicks)      AS clk_prev90,
           AVG(gsc_avg_position) AS pos_avg_prev90,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_prev90,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_active_prev90
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev90 >= 100
""").df()

print(f"{len(feat):,} rows, 5 corrected features (all from fact_daily, verified month-aligned)")
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,372 rows, 5 corrected features (all from fact_daily, verified month-aligned)


,client_hash_id,content_hash_id,imp_prev90,clk_prev90,pos_avg_prev90,ctr_prev90,days_active_prev90
0,client_62f4a7e64f5e0096,content_ae725e6f1852a254,228.0,0.0,65.287814,0.000000,65
1,client_62f4a7e64f5e0096,content_48e8152390b84fa6,14386.0,36.0,3.587036,0.002502,91
2,client_62f4a7e64f5e0096,content_de62c7692feb64a6,16269.0,29.0,2.115424,0.001783,91
3,client_62f4a7e64f5e0096,content_1a834c5cb078328d,15595.0,12.0,4.179759,0.000769,91
4,client_62f4a7e64f5e0096,content_e17c9534314cd3d2,518.0,1.0,10.544365,0.001931,88


Now **the deliberate leak**: I add a column computed from the label window itself (imp_next30, the very thing I'm trying to predict), watch the quick score jump toward perfect, then delete it.

In [10]:
# Build the future label window (kept separate — this is the label, not a feature)
label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1, 2
""").df()

leak_demo = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')   # <-- changed from feature_frame to feat
leak_demo['is_declining_future'] = (leak_demo['imp_next30'] < 0.8 * (leak_demo['imp_prev90'] / 3)).astype(int)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ['imp_prev90', 'clk_prev90', 'pos_avg_prev90', 'ctr_prev90', 'days_active_prev90']
leak_cols = honest_cols + ['imp_next30']

data_leak = leak_demo.dropna(subset=leak_cols + ['is_declining_future'])
X_tr, X_te, y_tr, y_te = train_test_split(
    data_leak[leak_cols], data_leak['is_declining_future'], test_size=0.25, random_state=42)
model_leak = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
print(f"WITH leak (imp_next30 included): AUC = {roc_auc_score(y_te, model_leak.predict_proba(X_te)[:,1]):.3f}")

data_honest = leak_demo.dropna(subset=honest_cols + ['is_declining_future'])
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    data_honest[honest_cols], data_honest['is_declining_future'], test_size=0.25, random_state=42)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr2, y_tr2)
print(f"WITHOUT leak (honest features only): AUC = {roc_auc_score(y_te2, model_honest.predict_proba(X_te2)[:,1]):.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WITH leak (imp_next30 included): AUC = 0.999
WITHOUT leak (honest features only): AUC = 0.699


Three numbers, one lesson: AUC = 0.999 with the deliberate imp_next30 leak, 0.841 with visible_queries/rare_share still included (the hidden fixed-window leak caught in Query D), and 0.706 with only genuinely month-aligned, decision-time-safe features.

The gap between 0.841 and 0.706 is the cost of a leak that wasn't obvious from the schema alone — it only surfaced by explicitly checking window_start/window_end. 0.706 is the number I'd actually defend.

## 4. Data limits

- Unbalanced panel: client history depth varies — only 9 of 70 clients have 12+ months of GSC history, so seasonality-dependent conclusions can't be drawn evenly across clients.
- GSC-only early rows: rows before a client's ga4_data_start have ga4_data_available = FALSE, meaning any engagement/session feature is silently unavailable for part of the panel, not zero.
- Freshness lag: the daily fact table stops at 2026-06-30 (freshest 3 days cut on purpose), so nothing in this release can speak to the most recent few days.
- Sealed test month: fact_daily_sample covers only June 2026, the natural outcome window of any past→future label — it's reserved as a sealed test month, never used to develop label logic.
- Window overlap risk: any feature accidentally computed from dates inside 2026-04-01–2026-04-30 would leak into this contract's label — Section 3's trap demonstrates exactly this failure mode and why it's checked for every lane, every time.
- Fixed-window leak in the query-mix table: fact_content_query_90d is not month-partitioned — it's a single snapshot with window_start = 2026-04-02, window_end = 2026-06-30 (confirmed by direct query). For any anchor month before April 2026, its fields are unusable as features without leaking future information. This was caught only by explicitly checking the window bounds, not by inspection of the schema alone — exactly the kind of leak Section 12 of the lane guide warns is easy to miss.

In [11]:
clients_hist = con.sql(f"""
    SELECT COUNT(*) AS n_clients,
           SUM(CASE WHEN gsc_data_start <= '2025-06-30' THEN 1 ELSE 0 END) AS clients_12mo_plus
    FROM {TABLES['dim_clients']}
""").df()
print(clients_hist)

   n_clients  clients_12mo_plus
0        104                9.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.